In [1]:
from utilities import save_dataset, retrieve_datasets

save_dataset()
dataset, train_dataset, val_dataset, test_dataset = retrieve_datasets()

print(dataset)
print("\nColumn names:", dataset["train"].column_names)
print("Total Number of examples:", len(dataset["train"]))
print("Train Number of examples:", len(train_dataset))
print("Validation Number of examples:", len(val_dataset))
print("Test Number of examples:", len(test_dataset))


/Users/dipeshpaneru/Documents/UniveristyWork/Advanced AI/Course-Assistant/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'subject', 'reference_answer'],
        num_rows: 651840
    })
})

Column names: ['question', 'answer', 'subject', 'reference_answer']
Total Number of examples: 651840
Train Number of examples: 586656
Validation Number of examples: 32592
Test Number of examples: 32592


In [2]:
from models import get_base_model
base_model, tokenizer = get_base_model()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/threading.py", line 1009, in _bootstrap_inner
    self.run()
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/threading.py", line 946, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/dipeshpaneru/Documents/UniveristyWork/Advanced AI/Course-Assistant/venv/lib/python3.10/site-packages/transformers/safetensors_conversion.py", line 117, in auto_conversion
    raise e
  File "/Users/dipeshpaneru/Documents/UniveristyWork/Advanced AI/Course-Assistant/venv/lib/python3.10/site-packages/transformers/safetensors_conversion.py", line 96, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
  File "/Users/dipeshpaneru/Documents/UniveristyWork/Advanced AI/Course-Assistant/venv/lib/python3.

Model loaded!
Device: mps:0


In [ ]:
from evaluation import Evaluation
import json
from datetime import datetime
import os


#Getting a subset from the test 
test_subset  = test_dataset.shuffle(seed=42).select(range(200))

evaluator = Evaluation()
collected = evaluator.collect_outputs(base_model, tokenizer, test_subset, stage_name="baseline")  #Infer the output from model




Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

In [4]:
results = evaluator.evaluate_results(base_model, tokenizer, collected) # Get the result of various matrices for each oputput

In [5]:

results = evaluator.human_eval(results)


HUMAN EVALUATION - Sample of 15 examples
QUESTION: What is the total sample size needed to compare the proportions of black and white teenagers without physical checkups within the last two years, given that the proportions are 17% and 7%, respectively? The study aims to detect a 10% difference with 90% power using a two-sided statistical test at the 0.01 level of significance.
MODEL OUTPUT:
The sample consists of 246 respondents who have completed their studies in 5th year medical school between January – May 2022.
We will use a mixed quantitative research methodology. We conducted semi structured interviews where we discussed the topic, and we took notes on our findings during the interview process. Then, we analysed the data through analysis of the interview transcript by a single examiner using inductive content analysis (ICA) methods which was developed by Grounded Theory approach [38]. ICA allows us to identify patterns across text fragments so as to build up a coherent narrativ

In [6]:
summary = evaluator.compute_averages(results)


📊 Baseline Evaluation Summary
  Perplexity:      6.72
  ROUGE-1:         0.1567
  ROUGE-2:         0.0178
  ROUGE-L:         0.0934
  Repetition Rate: 0.0558
  Repetition Rate higher than 1.5: 25
  Question Count:  1.66
  Human Evaluation:  1.27


In [7]:
from utilities import save_logs
save_logs("baseline", results, summary)


✅ Logs saved to outputs/


In [ ]:
#------------- Fine-tuned model evaluation -------------

from models import get_fine_tuned_model
ft_model, ft_tokenizer = get_fine_tuned_model()

In [ ]:
from evaluation import Evaluation

# Use the same 200-example test subset as baseline (same seed for fair comparison)
test_subset = test_dataset.shuffle(seed=42).select(range(200))

ft_evaluator = Evaluation()
ft_collected = ft_evaluator.collect_outputs(ft_model, ft_tokenizer, test_subset, stage_name="finetuned")
print(f"✅ Collected {len(ft_collected)} outputs")

In [ ]:
# Compute all metrics
ft_results = ft_evaluator.evaluate_results(ft_model, ft_tokenizer, ft_collected)

In [ ]:
# Human evaluation on same 15 examples as baseline
ft_results = ft_evaluator.human_eval(ft_results)

In [ ]:
ft_summary = ft_evaluator.compute_averages(ft_results)

In [2]:
import json
from evaluation import Evaluation

with open('../outputs/finetuned_full_20260408_185637.json') as f: 
    ft_results = json.load(f)

ft_results = Evaluation().human_eval(ft_results)




HUMAN EVALUATION - Sample of 15 examples
QUESTION: What is lymphosarcoma in fish, and what diagnostic methods are used to confirm it?
MODEL OUTPUT:
The 2016–17 influenza season was the most severe on record. This study aims to determine whether influenza A(H3N2) subtypes have contributed significantly or not to this increase compared with previous seasons of similar severity (e.g., ≥95% confidence interval). The data analyzed include: - Confirmed influenza cases by age group for all countries during January-September; - Age groups covered from children upward through adults; - Previous years' data for comparisons; - The 8 H3N2 influenzas identified as contributing to the recent outbreaks. Key findings include:   * Influenzavirus B dominated the virus pool over time but declined after April 2017.    * An additional "X" strain emerged that caused an epidemic among pregnant women at higher risk than normal populations. These results indicate both
Score each dimension 1-5:
  Relevance  — 

ValueError: invalid literal for int() with base 10: ''